# 02. Autonomous Context Updating 실습

## 학습 목표

- 긴 연구 기록에서 다음 결정에 필요한 상태만 추출합니다.
- 검증된 사실과 출처, 기각 후보와 이유, 미해결 제약, 다음 계획을 보존합니다.
- 압축 뒤에도 반드시 유지돼야 할 invariant를 테스트합니다.

이 예제는 문자열 길이를 token budget의 간단한 대용치로 사용합니다. 실제 token 수나 논문 모델의 ACU 품질을 재현하는 것은 아닙니다.

In [ ]:
from dataclasses import dataclass, field, asdict
import json

@dataclass(frozen=True)
class Event:
    kind: str
    claim: str
    source_id: str | None = None
    reason: str | None = None

@dataclass
class ImprovementState:
    verified: dict[str, str] = field(default_factory=dict)
    candidates: list[str] = field(default_factory=list)
    rejected: dict[str, str] = field(default_factory=dict)
    unresolved: list[str] = field(default_factory=list)
    validity_concerns: list[str] = field(default_factory=list)
    next_plan: list[str] = field(default_factory=list)

events = [
    Event("search", "후보 A와 후보 B를 발견"),
    Event("candidate", "후보 A"),
    Event("candidate", "후보 B"),
    Event("verified", "후보 A는 2026년에 공개", "src-arxiv"),
    Event("noise", "같은 검색 결과를 다시 읽음"),
    Event("rejected", "후보 B", reason="parameter 규모가 조건을 초과"),
    Event("unresolved", "후보 A의 weight license"),
    Event("concern", "블로그 날짜와 모델 카드 날짜가 다름"),
    Event("plan", "공식 모델 카드에서 license를 확인"),
]

In [ ]:
def consolidate(history: list[Event]) -> ImprovementState:
    """논문이 열거한 ACU 필드를 규칙 기반으로 구성합니다."""
    state = ImprovementState()
    for event in history:
        if event.kind == "verified":
            # 검증 사실은 source ID 없이 저장하면 나중에 재검증할 수 없습니다.
            if not event.source_id:
                raise ValueError("verified event에는 source_id가 필요합니다")
            state.verified[event.claim] = event.source_id
        elif event.kind == "candidate":
            state.candidates.append(event.claim)
        elif event.kind == "rejected":
            state.rejected[event.claim] = event.reason or "이유 미기록"
        elif event.kind == "unresolved":
            state.unresolved.append(event.claim)
        elif event.kind == "concern":
            state.validity_concerns.append(event.claim)
        elif event.kind == "plan":
            state.next_plan.append(event.claim)
        # search/noise 이벤트는 결정에 필요한 새 정보가 없으므로 버립니다.

    # 기각된 후보는 현재 후보 목록에서 제거하되 기각 이유는 보존합니다.
    state.candidates = [c for c in state.candidates if c not in state.rejected]
    return state

state = consolidate(events)
print(json.dumps(asdict(state), ensure_ascii=False, indent=2))

In [ ]:
def validate_state(state: ImprovementState, require_unresolved: bool = True) -> None:
    """압축이 중요한 정보를 잃지 않았는지 검사하는 회귀 테스트입니다."""
    assert state.verified, "검증 사실이 사라졌습니다"
    assert all(state.verified.values()), "source ID가 빠졌습니다"
    if require_unresolved:
        assert state.unresolved, "진행 중 연구의 미해결 제약이 사라졌습니다"
    assert state.next_plan, "다음 계획이 없습니다"
    assert not (set(state.candidates) & set(state.rejected)), "기각 후보가 활성 후보에 남았습니다"

validate_state(state)

raw = json.dumps([asdict(e) for e in events], ensure_ascii=False)
compact = json.dumps(asdict(state), ensure_ascii=False)
print("원 기록 문자 수:", len(raw))
print("갱신 상태 문자 수:", len(compact))
print("단순 압축률:", f"{1 - len(compact) / len(raw):.1%}")

## 갱신 이후 새 관찰 결합

논문의 유효 컨텍스트는 가장 최근의 압축 상태와 그 이후 새 상호작용을 이어 붙입니다. 이미 기각한 후보의 반복 검색은 피하고 미해결 조건에 직접 답하는 관찰만 반영해 봅니다.

In [ ]:
new_events = [
    Event("verified", "후보 A의 weight license는 허용됨", "src-model-card"),
]

refreshed = consolidate(events + new_events)
refreshed.unresolved = [
    item for item in refreshed.unresolved if item != "후보 A의 weight license"
]
refreshed.next_plan = ["모든 제약을 다시 감사하고 구조화된 답을 만든다"]
validate_state(refreshed, require_unresolved=False)
print(json.dumps(asdict(refreshed), ensure_ascii=False, indent=2))

assert "후보 B" in refreshed.rejected
assert "후보 A의 weight license는 허용됨" in refreshed.verified

## 확장 과제

1. source ID에 URL, 확인 날짜, 직접 인용 범위를 추가합니다.
2. 서로 모순되는 `verified` 이벤트가 들어오면 confidence를 낮추는 규칙을 만듭니다.
3. 웹 문서의 prompt injection 문구가 `next_plan`에 들어가지 않도록 입력과 정책 명령을 분리합니다.
4. 압축 전후의 claim-source 연결이 유지되는지 property-based test를 작성합니다.